# As estatísticas da aula 2 em SQL

## Task 2: traduzir as estatísticas da aula 2 para SQL e guardar a consulta no projeto

As estatísticas da aula 2 em SQL, com resultado **idêntico** ao da DataFrame API.

A célula abaixo já vem pronta: o mesmo código do vídeo 3.1, com as cinco views do projeto.

In [ ]:
# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
    )
)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

## Gols por Copa em SQL

## A tabela por seleção: CTE e UNION ALL

- **CTE** (`WITH nome AS (...)`): uma consulta intermediária com nome.
- **`UNION ALL`**: empilha os resultados de duas consultas.

### `UNION` não é `UNION ALL`

`UNION` remove linhas duplicadas; `UNION ALL` mantém todas. Com `UNION`, dois placares iguais da mesma seleção na
mesma Copa virariam um só.

## Provando que o resultado é idêntico ao da aula 2

`exceptAll`: as linhas da primeira tabela que não estão na segunda. Zero nos dois sentidos = tabelas iguais (as
colunas são comparadas pela posição).

## A auditoria em SQL

`LEFT ANTI JOIN`: as linhas da esquerda sem par na direita.

Zero, como na aula 2. No SQL padrão: `WHERE NOT EXISTS`.

## Guardando a consulta no projeto

Consultas reaproveitadas ficam em arquivos `.sql`, versionados com o código.

## SQL ou DataFrame API?

| Use SQL quando... | Use a DataFrame API quando... |
|---|---|
| a consulta é declarativa e longa (vários joins, CTEs) | há lógica de programa (laços, parâmetros, condições) |
| quem vai ler ou manter conhece SQL | você quer reaproveitar a transformação como função |
| é uma consulta de análise, para explorar | a transformação precisa de teste automatizado (aula 5) |

## Resultado da task 2

Estatísticas em SQL idênticas às da aula 2, e a consulta em `sql/gols_por_copa.sql`. Próxima task: os rankings.

## Para consultar depois

- **A armadilha do `NOT IN`:** `WHERE x NOT IN (SELECT ...)` devolve **vazio** se a subconsulta tiver um único `NULL`,
  porque, para o SQL, comparar com `NULL` não dá verdadeiro nem falso. Para "o que está em A e não está em B", prefira
  `LEFT ANTI JOIN` ou `NOT EXISTS`.
- **`LEFT SEMI JOIN`:** as linhas da esquerda que **têm** par na direita, como um `WHERE EXISTS`.
- **Joins em SQL** seguem a sintaxe que você conhece: `FROM partidas p JOIN copas c ON p.ano = c.ano`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()